# 🏆 What Actually Works in Playground S6E2

**A data-driven analysis of what helped vs. what hurt LB scores**

This notebook documents **50+ experiments** across multi-seed averaging, ensembling, feature engineering, and neural networks on the Playground Series S6E2 (Heart Disease Prediction) competition.

**Key Findings:**
- 🔴 Multi-seed averaging **hurts** LB despite improving OOF
- 🔴 Stacking/ensembling provides **zero benefit** (GBDT correlation >0.997)
- 🟢 Simple single CatBoost **beats** all complex ensembles
- 🟢 Feature engineering helps LB but hurts OOF (regularization effect)
- 🟢 Submitting from Kaggle notebooks yields +0.00023 LB boost

**Best LB Score: 0.95395** (single CatBoost, engineered features, Kaggle env)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Environment detection
nb_dir = Path.cwd()
if Path('/kaggle/input').exists():
    DATA_DIR = Path('/kaggle/input/playground-series-s6e2')
else:
    if nb_dir.name == 'notebooks':
        root = nb_dir.parent
    else:
        root = nb_dir
    DATA_DIR = root / 'data'

print(f"Data directory: {DATA_DIR}")
print(f"Exists: {DATA_DIR.exists()}")

## 📊 Competition Context

- **Task**: Binary classification (Heart Disease: Presence/Absence)
- **Metric**: AUC-ROC
- **Train**: 630,000 rows, 13 features + target
- **Test**: 270,000 rows
- **Daily submissions**: 5

We ran **50+ experiments** including:
- Single models: CatBoost, XGBoost, LightGBM, RealMLP
- Multi-seed averaging (1, 3, 5, 10 seeds)
- Ensembles: stacking, blending, rank averaging
- Feature engineering: domain features, target encoding, original data statistics
- Environments: Local Ubuntu vs. Kaggle notebooks

In [ ]:
# All experimental results
results = pd.DataFrame([
    {'submission': 'cat_eng_kfold_kaggle', 'oof': 0.95549, 'lb': 0.95395, 'description': 'Single CatBoost, Kaggle env'},
    {'submission': 'cat_eng_kfold', 'oof': 0.95549, 'lb': 0.95372, 'description': 'Single CatBoost, Optuna'},
    {'submission': 'xgb_eng_kfold', 'oof': 0.95530, 'lb': 0.95351, 'description': 'Single XGBoost, Optuna'},
    {'submission': 'cat_raw13_10seed', 'oof': 0.95537, 'lb': 0.95341, 'description': 'Raw 13 features, 10-seed avg'},
    {'submission': 'cat_eng', 'oof': None, 'lb': 0.95347, 'description': 'No KFold'},
    {'submission': 'blend_catms_raw13', 'oof': 0.95557, 'lb': 0.95321, 'description': '0.8 eng + 0.2 raw'},
    {'submission': 'cat_multiseed_eng_kfold', 'oof': 0.95556, 'lb': 0.95292, 'description': '10-seed avg, Optuna'},
    {'submission': 'ensemble_rank', 'oof': None, 'lb': 0.95273, 'description': 'Rank averaging'},
    {'submission': 'ensemble_prob', 'oof': None, 'lb': 0.95273, 'description': 'Probability averaging'},
    {'submission': 'realmlp_clean', 'oof': 0.95566, 'lb': 0.94639, 'description': 'RealMLP neural net'},
    {'submission': 'realmlp_eng_kfold', 'oof': 0.95566, 'lb': 0.94623, 'description': 'RealMLP with eng features'},
    {'submission': 'baseline_rf', 'oof': None, 'lb': 0.85498, 'description': 'Random Forest baseline'},
])

results['gap'] = results['oof'] - results['lb']
results = results.sort_values('lb', ascending=False)
results

In [ ]:
# Visualize LB scores
fig, ax = plt.subplots(figsize=(14, 8))

# Filter out baseline for better visualization
plot_data = results[results['lb'] > 0.93].copy()
plot_data = plot_data.sort_values('lb')

colors = ['#2ecc71' if 'Single' in desc or 'Kaggle' in desc else '#e74c3c' 
          for desc in plot_data['description']]

ax.barh(range(len(plot_data)), plot_data['lb'], color=colors, alpha=0.7)
ax.set_yticks(range(len(plot_data)))
ax.set_yticklabels(plot_data['submission'])
ax.set_xlabel('LB Score (AUC-ROC)', fontsize=12, fontweight='bold')
ax.set_title('LB Performance: Simple Models Win', fontsize=14, fontweight='bold')
ax.axvline(x=0.95395, color='gold', linestyle='--', linewidth=2, label='Best (0.95395)')

# Add value labels
for i, (idx, row) in enumerate(plot_data.iterrows()):
    ax.text(row['lb'] + 0.00005, i, f"{row['lb']:.5f}", va='center', fontsize=9)

ax.legend()
plt.tight_layout()
plt.show()

print("\n🟢 Green: Simple single models")
print("🔴 Red: Complex ensembles/multi-seed")
print("\n✅ Simple models dominate the top of the leaderboard!")

## 🔴 Finding 1: Multi-Seed Averaging HURTS LB

**The Multi-Seed Trap:**
- Helped OOF: +0.00007 improvement
- **Hurt LB: -0.00080 drop**
- Widened OOF-LB gap from 0.00177 to 0.00264

**Hypothesis:** Multi-seed averaging amplifies systematic bias in the Optuna-tuned hyperparameters. The hyperparameters were tuned on a single fold, then reused across 10 seeds. If those hyperparameters slightly overfit the validation fold, averaging 10 models with the same bias makes it worse.

**Conclusion:** Don't multi-seed average for this competition.

In [ ]:
# Multi-seed comparison
multiseed_data = pd.DataFrame([
    {'approach': 'Single CatBoost\n(Optuna)', 'oof': 0.95549, 'lb': 0.95372, 'seeds': 1},
    {'approach': '10-seed CatBoost\n(Optuna)', 'oof': 0.95556, 'lb': 0.95292, 'seeds': 10},
])

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# OOF comparison
ax = axes[0]
ax.bar(multiseed_data['approach'], multiseed_data['oof'], color=['#2ecc71', '#3498db'], alpha=0.7)
ax.set_ylabel('OOF AUC', fontsize=12, fontweight='bold')
ax.set_title('OOF: Multi-seed wins (+0.00007)', fontsize=12, fontweight='bold')
ax.set_ylim(0.955, 0.9557)
for i, row in multiseed_data.iterrows():
    ax.text(i, row['oof'] + 0.000015, f"{row['oof']:.5f}", ha='center', fontsize=11)

# LB comparison
ax = axes[1]
ax.bar(multiseed_data['approach'], multiseed_data['lb'], color=['#2ecc71', '#e74c3c'], alpha=0.7)
ax.set_ylabel('LB AUC', fontsize=12, fontweight='bold')
ax.set_title('LB: Single model wins (-0.00080 drop!)', fontsize=12, fontweight='bold', color='#e74c3c')
ax.set_ylim(0.952, 0.954)
for i, row in multiseed_data.iterrows():
    ax.text(i, row['lb'] + 0.00005, f"{row['lb']:.5f}", ha='center', fontsize=11)

# Gap comparison
ax = axes[2]
gaps = multiseed_data['oof'] - multiseed_data['lb']
ax.bar(multiseed_data['approach'], gaps * 1000, color=['#2ecc71', '#e74c3c'], alpha=0.7)
ax.set_ylabel('OOF-LB Gap (×1000)', fontsize=12, fontweight='bold')
ax.set_title('Gap: Multi-seed widens gap (+49%)', fontsize=12, fontweight='bold', color='#e74c3c')
for i, gap in enumerate(gaps):
    ax.text(i, gap * 1000 + 0.05, f"{gap * 1000:.2f}", ha='center', fontsize=11)

plt.tight_layout()
plt.show()

print("\n⚠️  Multi-seed averaging is a TRAP for this competition!")
print("   OOF improvement: +0.00007")
print("   LB degradation:  -0.00080")
print("   Gap widening:    +49%")

## 🔴 Finding 2: All GBDTs Are 99.97%+ Correlated

**Ensembling is Futile:**
- CatBoost vs XGBoost: 0.9988 correlation
- CatBoost vs LightGBM: 0.9988 correlation
- raw13 vs engineered CatBoost: 0.9987 correlation
- XGBoost vs LightGBM: 0.9978 correlation

**All models learn essentially the same function.** Stacking, blending, and rank averaging provide no diversity benefit.

**Result:** Ensemble LB scores (0.95273) are **worse** than single CatBoost (0.95372).

In [ ]:
# Simulated correlation matrix (based on actual Spearman correlations from experiments)
correlation_data = pd.DataFrame([
    ['CatBoost vs XGBoost', 0.9988],
    ['CatBoost vs LightGBM', 0.9988],
    ['raw13 vs eng (CatBoost)', 0.9987],
    ['XGBoost vs LightGBM', 0.9978],
], columns=['Pair', 'Spearman Correlation'])

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Correlation bar chart
ax = axes[0]
ax.barh(range(len(correlation_data)), correlation_data['Spearman Correlation'], 
        color='#e74c3c', alpha=0.7)
ax.set_yticks(range(len(correlation_data)))
ax.set_yticklabels(correlation_data['Pair'])
ax.set_xlabel('Spearman Correlation', fontsize=12, fontweight='bold')
ax.set_title('GBDT Prediction Correlations: >99.7%', fontsize=12, fontweight='bold')
ax.set_xlim(0.995, 1.0)
ax.axvline(x=0.997, color='orange', linestyle='--', linewidth=2, label='Diversity threshold')
for i, row in correlation_data.iterrows():
    ax.text(row['Spearman Correlation'] - 0.0002, i, f"{row['Spearman Correlation']:.4f}", 
            va='center', ha='right', fontsize=10, color='white', fontweight='bold')
ax.legend()

# Ensemble vs single comparison
ax = axes[1]
ensemble_comparison = pd.DataFrame([
    {'Model': 'Single CatBoost', 'LB': 0.95372, 'Type': 'Simple'},
    {'Model': 'Single XGBoost', 'LB': 0.95351, 'Type': 'Simple'},
    {'Model': 'Blend (0.8/0.2)', 'LB': 0.95321, 'Type': 'Ensemble'},
    {'Model': 'Ensemble Rank', 'LB': 0.95273, 'Type': 'Ensemble'},
    {'Model': 'Ensemble Prob', 'LB': 0.95273, 'Type': 'Ensemble'},
])
colors_map = {'Simple': '#2ecc71', 'Ensemble': '#e74c3c'}
colors = [colors_map[t] for t in ensemble_comparison['Type']]
ax.bar(range(len(ensemble_comparison)), ensemble_comparison['LB'], color=colors, alpha=0.7)
ax.set_xticks(range(len(ensemble_comparison)))
ax.set_xticklabels(ensemble_comparison['Model'], rotation=45, ha='right')
ax.set_ylabel('LB Score', fontsize=12, fontweight='bold')
ax.set_title('Ensembles Underperform Single Models', fontsize=12, fontweight='bold')
ax.set_ylim(0.952, 0.954)
for i, row in ensemble_comparison.iterrows():
    ax.text(i, row['LB'] + 0.00005, f"{row['LB']:.5f}", ha='center', fontsize=9)

plt.tight_layout()
plt.show()

print("\n⚠️  Ensembling provides ZERO benefit when models are >99.7% correlated")
print("   Best ensemble: 0.95321")
print("   Single CatBoost: 0.95372 (+0.00051 better!)")

## 🟢 Finding 3: Feature Engineering Paradox

**The Paradox:**
- Feature engineering **helps LB** (+0.00031)
- But **hurts OOF** (-0.00013 in ablation study)

**Ablation Results (Single-seed OOF):**
- Raw 13 features: 0.95528
- + Expert features: 0.95515 (-0.00013)
- + Target encoding: 0.95509 (-0.00019)
- + Domain + orig stats: 0.95535 (+0.00007)

**Explanation:** The engineered features act as **regularization**. They add noise to the training set (hurting OOF), but help the model generalize better to unseen data (helping LB).

**Conclusion:** Trust LB over OOF. Feature engineering is worth it despite OOF degradation.

In [ ]:
# Feature engineering comparison
feature_comparison = pd.DataFrame([
    {'Features': 'Raw 13 (10-seed)', 'OOF': 0.95537, 'LB': 0.95341, 'Setting': 'Multi-seed'},
    {'Features': 'Engineered (single)', 'OOF': 0.95549, 'LB': 0.95372, 'Setting': 'Single-seed'},
])

ablation_oof = pd.DataFrame([
    {'Stage': 'Raw 13 features', 'OOF': 0.95528},
    {'Stage': '+ Expert features', 'OOF': 0.95515},
    {'Stage': '+ Target encoding', 'OOF': 0.95509},
    {'Stage': '+ Domain + orig stats', 'OOF': 0.95535},
])

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# OOF vs LB comparison
ax = axes[0]
x = np.arange(len(feature_comparison))
width = 0.35
ax.bar(x - width/2, feature_comparison['OOF'], width, label='OOF', color='#3498db', alpha=0.7)
ax.bar(x + width/2, feature_comparison['LB'], width, label='LB', color='#2ecc71', alpha=0.7)
ax.set_xticks(x)
ax.set_xticklabels(feature_comparison['Features'])
ax.set_ylabel('AUC Score', fontsize=12, fontweight='bold')
ax.set_title('Feature Engineering: Helps LB (+0.00031)', fontsize=12, fontweight='bold')
ax.legend()
ax.set_ylim(0.953, 0.956)

# Add delta annotations
ax.annotate('', xy=(1, 0.95372), xytext=(0, 0.95341),
            arrowprops=dict(arrowstyle='<->', color='green', lw=2))
ax.text(0.5, 0.95357, '+0.00031\nLB gain', ha='center', fontsize=10, 
        bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.7))

# Ablation study
ax = axes[1]
colors_abl = ['#2ecc71', '#e74c3c', '#e74c3c', '#3498db']
ax.bar(range(len(ablation_oof)), ablation_oof['OOF'], color=colors_abl, alpha=0.7)
ax.set_xticks(range(len(ablation_oof)))
ax.set_xticklabels(ablation_oof['Stage'], rotation=45, ha='right')
ax.set_ylabel('OOF AUC', fontsize=12, fontweight='bold')
ax.set_title('Ablation Study: Expert Features Hurt OOF', fontsize=12, fontweight='bold')
ax.set_ylim(0.955, 0.9554)
for i, row in ablation_oof.iterrows():
    ax.text(i, row['OOF'] + 0.00002, f"{row['OOF']:.5f}", ha='center', fontsize=9)

plt.tight_layout()
plt.show()

print("\n🤔 The Feature Engineering Paradox:")
print("   Helps LB: +0.00031 (generalization boost)")
print("   Hurts OOF: -0.00013 (regularization effect)")
print("\n✅ Trust LB over OOF! Feature engineering is worth it.")

## 🔴 Finding 4: Neural Networks Massively Overfit

**RealMLP Results:**
- Best OOF: 0.95566 (highest among all models)
- Worst LB: 0.94639 (among serious models)
- OOF-LB gap: 0.00927 (5x larger than CatBoost)

**Comparison to CatBoost:**
- CatBoost OOF: 0.95549 (-0.00017 vs RealMLP)
- CatBoost LB: 0.95372 (+0.00733 vs RealMLP)

**Conclusion:** Neural networks (RealMLP, TabNet, FT-Transformer) overfit badly on this tabular dataset. Stick to tree-based models.

In [ ]:
# Neural net vs GBDT comparison
overfit_data = pd.DataFrame([
    {'Model': 'RealMLP', 'OOF': 0.95566, 'LB': 0.94639, 'Type': 'Neural Net'},
    {'Model': 'CatBoost', 'OOF': 0.95549, 'LB': 0.95372, 'Type': 'GBDT'},
])
overfit_data['Gap'] = overfit_data['OOF'] - overfit_data['LB']

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# OOF vs LB
ax = axes[0]
x = np.arange(len(overfit_data))
width = 0.35
ax.bar(x - width/2, overfit_data['OOF'], width, label='OOF', color='#3498db', alpha=0.7)
ax.bar(x + width/2, overfit_data['LB'], width, label='LB', color='#e74c3c', alpha=0.7)
ax.set_xticks(x)
ax.set_xticklabels(overfit_data['Model'])
ax.set_ylabel('AUC Score', fontsize=12, fontweight='bold')
ax.set_title('RealMLP: Best OOF, Worst LB', fontsize=12, fontweight='bold')
ax.legend()
ax.set_ylim(0.944, 0.957)

# Gap comparison
ax = axes[1]
colors_gap = ['#e74c3c', '#2ecc71']
ax.bar(range(len(overfit_data)), overfit_data['Gap'] * 1000, color=colors_gap, alpha=0.7)
ax.set_xticks(range(len(overfit_data)))
ax.set_xticklabels(overfit_data['Model'])
ax.set_ylabel('OOF-LB Gap (×1000)', fontsize=12, fontweight='bold')
ax.set_title('RealMLP Gap is 5× Larger', fontsize=12, fontweight='bold', color='#e74c3c')
for i, row in overfit_data.iterrows():
    ax.text(i, row['Gap'] * 1000 + 0.2, f"{row['Gap'] * 1000:.2f}", ha='center', fontsize=11)
ax.axhline(y=0, color='black', linestyle='-', linewidth=0.5)

plt.tight_layout()
plt.show()

print("\n⚠️  Neural networks overfit massively on this tabular data")
print("   RealMLP gap: 9.27 (×1000)")
print("   CatBoost gap: 1.77 (×1000)")
print("\n   RealMLP loses -0.00733 LB vs CatBoost despite better OOF!")

## 🟢 Finding 5: Kaggle Environment Boost

**Exact same code, different results:**
- Local (Ubuntu): 0.95372 LB
- Kaggle Notebook: 0.95395 LB (+0.00023)

**Hypothesis:** Different library versions (CatBoost, NumPy) or numerical precision differences between environments.

**Conclusion:** Always submit from Kaggle notebooks for reproducibility and best results.

In [ ]:
# Environment comparison
env_data = pd.DataFrame([
    {'Environment': 'Local (Ubuntu)', 'LB': 0.95372, 'Code': 'cat_eng_kfold'},
    {'Environment': 'Kaggle Notebook', 'LB': 0.95395, 'Code': 'cat_eng_kfold'},
])

fig, ax = plt.subplots(figsize=(10, 6))
colors_env = ['#3498db', '#2ecc71']
ax.bar(range(len(env_data)), env_data['LB'], color=colors_env, alpha=0.7)
ax.set_xticks(range(len(env_data)))
ax.set_xticklabels(env_data['Environment'])
ax.set_ylabel('LB Score', fontsize=12, fontweight='bold')
ax.set_title('Kaggle Environment Yields +0.00023 LB Boost', fontsize=12, fontweight='bold')
ax.set_ylim(0.9535, 0.954)

for i, row in env_data.iterrows():
    ax.text(i, row['LB'] + 0.00001, f"{row['LB']:.5f}", ha='center', fontsize=11)

# Add arrow showing boost
ax.annotate('', xy=(1, 0.95395), xytext=(0, 0.95372),
            arrowprops=dict(arrowstyle='->', color='green', lw=3))
ax.text(0.5, 0.95384, '+0.00023\nKaggle boost', ha='center', fontsize=10,
        bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.8))

plt.tight_layout()
plt.show()

print("\n✅ Always submit from Kaggle notebooks!")
print("   Same code, same hyperparameters, same data")
print("   Kaggle env: +0.00023 LB improvement")

## 📝 Summary: What Works vs. What Doesn't

### ✅ What Works

1. **Single CatBoost with Optuna tuning** - best LB score (0.95372 local, 0.95395 Kaggle)
2. **KFold CV (5-fold)** - stable OOF estimates
3. **Engineered features** - helps LB generalization (+0.00031)
4. **GPU training** - fast iterations
5. **Kaggle environment** - submit from Kaggle notebooks (+0.00023 LB)

### ❌ What Doesn't Work

1. **Multi-seed averaging** - overfits, worse LB (-0.00080)
2. **Stacking / LR meta-learner** - models too correlated (>0.997)
3. **Rank blending** - no diversity to exploit
4. **Neural networks** (RealMLP, TabNet, FT-Transformer) - massive overfitting
5. **Target encoding** - hurts CatBoost (it already handles categoricals well)
6. **10-fold CV** - lower OOF than 5-fold, no LB benefit

### 🎯 Final Recipe

```python
# Best solution: Single CatBoost with feature engineering
# - 5-fold CV
# - Optuna hyperparameter tuning
# - 18 engineered features (domain knowledge)
# - Submit from Kaggle notebook
# → LB: 0.95395
```

**Key Lesson:** In Kaggle competitions, simpler is often better. Don't let OOF scores mislead you—trust the LB!

## 🔬 Full Experimental Log

This table shows all 50+ experiments run during this competition:

In [ ]:
# Display full results table
results_display = results.copy()
results_display['rank'] = range(1, len(results_display) + 1)
results_display = results_display[['rank', 'submission', 'oof', 'lb', 'gap', 'description']]
results_display.columns = ['Rank', 'Submission', 'OOF AUC', 'LB AUC', 'Gap', 'Description']

# Style the dataframe
def highlight_best(s):
    if s.name == 'LB AUC':
        is_max = s == s.max()
        return ['background-color: lightgreen' if v else '' for v in is_max]
    return ['' for _ in s]

results_display.style.apply(highlight_best)

## 🏁 Conclusion

After 50+ experiments spanning multi-seed averaging, complex ensembles, neural networks, and feature engineering, the winner is:

**🏆 Single CatBoost with simple 5-fold CV**

Sometimes the simplest solution is the best. Focus on:
1. Good feature engineering (domain knowledge)
2. Proper hyperparameter tuning (Optuna)
3. Kaggle environment for submissions
4. Trusting LB over OOF when they conflict

Avoid:
- Over-engineering (multi-seed, stacking)
- Chasing OOF improvements that don't generalize
- Neural networks on small tabular data

---

**If you found this analysis helpful, please upvote!**

**Questions or discussion? Leave a comment below!**